In [ ]:
import sys
import os
import pickle
from tqdm.auto import tqdm

BASE_DIR = os.getcwd()
BASE_DIR

In [ ]:
# Set the primary font for visualization
from matplotlib import font_manager as fm

# Define the font path and initialize FontProperties
font_path = BASE_DIR + '/NotoSans-VariableFont_wdth,wght.ttf'
prop = fm.FontProperties(fname=font_path)

# Step 1: Preprocessing

## Step 1-1: Download and parsing

### Step 1-1-1: History dataset

1. Download English Wikipedia (enwiki) raw data from Wikimedia Dumps (https://dumps.wikimedia.org/) 

    Note: The version used is from Jan 23, 2025, but users may use any available version.

2. Extract specific features/elements from the raw data for analysis

#### 1. Data Acquisition (Terminal-based)

First, download the English Wikipedia stub-meta-history dumps.

We recommend creating a directory named raw_file/ to manage these tasks.
Due to the massive file sizes, we recommend processing them in chunks.

**Step A: Download and Decompress**

Download the stub-meta-history XML files (27 files in total).
(File size: Approx. 4 GB per file / Max 6.7 GB)

wget https://dumps.wikimedia.org/enwiki/20250123/enwiki-20250123-stub-meta-history1.xml.gz

Decompress the downloaded files.
Note: Ensure sufficient disk space before decompression (Approx. 24 GB per file, Max 44.4 GB).

gunzip enwiki-20250123-stub-meta-history1.xml.gz

**Step B: Data Splitting (Chunking)**

Since the files are too large for memory-efficient processing, split them into smaller segments.

Split into 10GB chunks:

split -b 10000M --numeric-suffixes enwiki-20250123-stub-meta-history1.xml history1_

Further split the segments into 1GB chunks for granular processing:

split -b 1000M --numeric-suffixes history1_00 history1_00_

Repeat the above process for all 27 stub-meta-history files.

#### 2. Feature Extraction from Raw Data (Preprocessing_history)

This step involves parsing the split XML chunks to extract specific elements (revisions, contributors, and timestamps) and saving them as structured .pickle files.

Input: Split XML files (processed sequentially by history_number).

Output: A dictionary saved as a pickle file.

Path: Dataset/raw_history/history_ns0_{history_number}.pickle

Structure: histories[page_id] = [page_title, [[timestamp, user_id_user_name], ...]]

In [ ]:
import html
import pickle
from collections import defaultdict
import re
import os
    
def Preprocessing_history(BASE_DIR, history_number, _10000M_number, _1000M_number_list):
    """
    Parses Wikipedia XML history dumps and saves them as structured pickle files.

    Args:
        BASE_DIR (str): Project root directory.
        history_number (int): ID for file naming.
        _10000M_number (int): Count of major groups.
        _1000M_number_list (list): Subgroup counts for each major group.
    """

    # Define directory paths for raw data and output targets
    DATA_DIR = os.path.join(BASE_DIR, 'raw_file')
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_history')
    
    # Automatically create the target directory if it does not exist
    if not os.path.exists(TARGET_DIR):
        os.makedirs(TARGET_DIR, exist_ok=True)
        print(f"[INFO] Created directory: {TARGET_DIR}")
        
    FileName = os.path.join(DATA_DIR, 'history')
    patterns = []
    for N_10000M in range(_10000M_number):
        patterns.append((FileName + str(history_number) + '_0' + str(N_10000M), _1000M_number_list[N_10000M]))

    # Generate file names based on the specified patterns
    file_names = []
    for prefix, count in patterns:
        for i in range(count):
            file_name = f'{prefix}_{i:02d}'  # Format the file name with zero-padding
            file_names.append(file_name)


    def process_xml_chunk(lines):
        """
        Parses a list of XML lines to extract page information and its revision history.

        Args:
            lines (list): A list of strings, where each string is a line from an XML file representing one or more <page> blocks.

        Returns:
            list: A list of extracted revision records [ns, page_id, title, redirect_title, timestamp, user_id, username].
        """
        # Convert list of lines into a single XML string
        xml_string = ''.join(lines)

        # Regular expression pattern to identify individual <page> blocks
        page_info_pattern = r'<page>.*?</page>'

        # Extract all page blocks from the XML string
        pages = re.findall(page_info_pattern, xml_string, re.DOTALL)
        all_rows = []

        for page in pages:
            # Extract the namespace (ns) of the page
            ns = re.search(r'<ns>(.*?)</ns>', page).group(1)

            # Extract the Page ID using a multi-line search pattern
            page_id_match = re.search(r'<title>.*?</title>.*?<ns>.*?</ns>.*?<id>(\d+)</id>', page, re.DOTALL)
            page_id = page_id_match.group(1) if page_id_match else None

            # Check for redirection and extract the redirect target title
            redirect_title = re.search(r'<redirect title="(.*?)"', page)
            
            # Extract the primary page title
            title = re.search(r'<title>(.*?)</title>', page).group(1)  # Use the standard title element
            if redirect_title:
                redi_title = redirect_title.group(1)  # Use title from the <redirect> tag
            else:
                redi_title = None

            # Extract revision history information
            revisions = re.findall(r'<revision>.*?</revision>', page, re.DOTALL)
            for revision in revisions:
                # Extract the timestamp of the revision
                timestamp = re.search(r'<timestamp>(.*?)</timestamp>', revision).group(1)

                # Extract contributor information (Username and User ID)
                username_match = re.search(r'<contributor>.*?<username>(.*?)</username>', revision, re.DOTALL)
                username = html.unescape(username_match.group(1)) if username_match else None

                user_id_match = re.search(r'<contributor>.*?<id>(\d+)</id>', revision, re.DOTALL)
                user_id = user_id_match.group(1) if user_id_match else None
                # Extract IP address for anonymous contributors
                ip_address_match = re.search(r'<contributor>.*?<ip>(.*?)</ip>', revision, re.DOTALL)
                ip_address = ip_address_match.group(1) if ip_address_match else None

                # Case 1: Both username and user_id are missing → Identify as Anonymous
                if username is None and user_id is None:
                    username = 'Anonymous'
                    user_id = ip_address if ip_address else 'ip'

                # Case 2: user_id is "0" → Not a valid unique identifier
                elif user_id == "0":
                    if username:
                        user_id = f"id_0_{username}"
                    elif ip_address:
                        user_id = f"id_0_{ip_address}"
                    else:
                        user_id = "id_0_unknown"
                # Aggregate all metadata into a single row for the dataset
                row = [ns, page_id, title, redi_title, timestamp, user_id, username]
                all_rows.append(row)
        return all_rows

    # Iterate through the file list and process each file
    all_rows = []
    for file_path in file_names:  # Provide the list of target files
        with open(file_path, 'r', encoding='utf-8') as file:
            buffer = []
            for line in file:
                buffer.append(line)
                if line.strip() == '</page>':
                    # Parse the current buffer and append the results to the main list
                    rows_chunk = process_xml_chunk(buffer)
                    all_rows.extend(rows_chunk)  # Merge the chunk results into all_rows
                    buffer = []  # Clear the buffer for the next page block

    # row = [ns, page_id, title, timestamp, user_id, username]
    # ns_list = [('-2', 'Media'), ('-1', 'Special'), ('0', '(Main/Article)'), ('1', 'Talk'), ('2', 'User'), ('3', 'User talk'), ('4', 'Wikipedia'), ('5', 'Wikipedia talk'), ('6', 'File'), ('7', 'File talk'), ('8', 'MediaWiki'), ('9', 'MediaWiki talk'), ('10', 'Template'), ('11', 'Template talk'), ('12', 'Help'), ('13', 'Help talk'), ('14', 'Category'), ('15', 'Category talk'), ('100', 'Portal'), ('101', 'Portal talk'), ('118', 'Draft'), ('119', 'Draft talk'), ('126', 'MOS'), ('127', 'MOS talk'), ('710', 'TimedText'), ('711', 'TimedText talk'), ('828', 'Module'), ('829', 'Module talk')]
    
    # Filter and group the extracted rows into a dictionary
    rows_dict = defaultdict(list)
    for row in all_rows:
        ns = str(row[0])
        if ns == '0':
            if row[3] == None:
                rows_dict[str(row[1])].append([row[2], row[4], row[5], row[6]])

    # Reconstruct the data into a history format for each page
    histories = {}
    for page in rows_dict.keys():
        rows = rows_dict[page]
        page_title = rows[0][0]
        histories[page] = [page_title]
        event_list = []
        for row in rows:
            user_id = row[2]
            user_name = row[3]
            timestamp = row[1]
            event_list.append([timestamp, str(user_id) + "_" + str(user_name)])
        # Append the list of revision events to the history dictionary
        histories[page].append(event_list)
        
    # Save the processed history using standardized file_path naming
    history_file_path = os.path.join(TARGET_DIR, f'history_ns0_{history_number}.pickle')
    with open(history_file_path, 'wb') as fw:
        pickle.dump(histories, fw)

    print(f"[FINISH] Processed history saved to: {history_file_path}")
    return

In [ ]:
# Configuration for data processing
history_number = 6            # The index number of the Wikipedia history dump file
_10000M_number = 3            # Total count of 10GB partitioned groups
_1000M_number_list = [10, 10, 4]  # List specifying the number of 1GB files within each 10GB group

# Execute the preprocessing pipeline
# This process handles a total of 24 files (sum of _1000M_number_list) for history_number 6
Preprocessing_history(BASE_DIR, history_number, _10000M_number, _1000M_number_list)

### Step 1-1-2: Hyperlink dataset

1. Download data from the Wikipedia dump site (as of Jan 23, 2025). (Refer to ### Download)

2. Extract specific elements from the raw data. (Refer to ### Extraction)

#### 1. Download (Terminal-based)

- page  
wget https://dumps.wikimedia.org/enwiki/20250123/enwiki-20250123-page.sql.gz  
gunzip enwiki-20250123-page.sql.gz  
==> Create 'page' table in MariaDB
- linkatrget  
wget https://dumps.wikimedia.org/enwiki/20250123/enwiki-20250123-linktarget.sql.gz  
gunzip enwiki-20250123-linktarget.sql.gz  
==> Create 'linktarget' table in MariaD
- redirect
wget https://dumps.wikimedia.org/enwiki/20250123/enwiki-20250123-redirect.sql.gz  
gunzip enwiki-20250123-redirect.sql.gz  
==> Create 'redirect' table in MariaD
- pagelinks (hyperlink info)  
wget https://dumps.wikimedia.org/enwiki/20250123/enwiki-20250123-pagelinks.sql.gz  
gunzip enwiki-20250123-pagelinks.sql.gz \
==> Manual parsing via Python (Recommended due to high overhead when uploading large files to MariaDB)

#### 2. Extract Pagelinks Information

This step parses the SQL dump to map source pages to their corresponding target hyperlinks, specifically filtering for the main article namespace.

Input: enwiki-20250123-pagelinks.sql

Processing Logic: * Extracts records where the namespace (pl_from_namespace) is 0 (Main/Article).

Filters and organizes the data into a dictionary for efficient lookup.

Output: file/from_target.pickle

Data Structure:

A dictionary where the key is the source page ID and the value is a list of target link IDs.

Format: from_target[from_id] = [pl_target_id, pl_target_id, ...]

Data Types: Key: str, Value: List[str]

In [ ]:
import pickle
import re
import os
from tqdm.auto import tqdm

def Parsing_pagelinks(BASE_DIR):
    """
    Parses Wikipedia 'pagelinks' SQL files in three steps: chunking, extraction, and merging.

    Step 1: Splits the large SQL file into smaller pickle chunks for memory efficiency.
    Step 2: Processes each chunk to extract 'from' and 'target' IDs for namespace 0 (articles).
    Step 3: Merges all processed chunks into a single unified 'from_target.pickle' file.

    Args:
        BASE_DIR (str): The root directory of the project.

    Returns:
        None: Saves the final merged dictionary to a pickle file.
    """
    
    DATA_DIR = os.path.join(BASE_DIR, "file")
    CHUNKS_DIR = os.path.join(DATA_DIR, "pagelinks_chunks") # Directory for intermediate files
    TARGET_DIR = DATA_DIR # Final output will be saved in the "file" directory
    
    # Create the directory for saving intermediate chunks
    os.makedirs(CHUNKS_DIR, exist_ok=True)

    # Define the source SQL file path (Input is from DATA_DIR)
    sql_file_path = os.path.join(DATA_DIR, "enwiki-20250123-pagelinks.sql")
    
    # Calculate the total number of lines to monitor progress
    with open(sql_file_path, "r", encoding="utf-8") as file:
        total_lines = sum(1 for _ in file)
    print(f"[INFO] Total lines in file: {total_lines}")
    # output: [INFO] Total lines in file: 34480
    
    def pagelinks_1st_step():
        """
        Reads the SQL file and saves it in chunks starting from a specific line.
        
        Saves every 2,000 lines into individual pickle files for easier processing.
        """
        chunk_size = 2000  # Number of lines to store per chunk
        chunk_index = 1  # Initial chunk index for file naming

        lines_to_save = []
        start_line = 40  # Processing starts from the 40th line

        with open(sql_file_path, "r", encoding="utf-8") as file:
            for i, line in enumerate(file, start=1):
                if i >= start_line:
                    lines_to_save.append(line.strip())
                    # Save when chunk size is reached or at the end of the file
                    if len(lines_to_save) == chunk_size or i == total_lines:
                        chunk_file_path = os.path.join(CHUNKS_DIR, f"chunk_{chunk_index}.pickle")
                        with open(chunk_file_path, "wb") as pkl_file:
                            pickle.dump(lines_to_save, pkl_file)

                        print(f"[LOG] Saved lines {start_line} to {i} in: {chunk_file_path}")

                        # Prepare for the next chunk
                        chunk_index += 1
                        lines_to_save = []

                if i >= total_lines:
                    break

        print("[SUCCESS] All chunks have been saved.")
        return
    # 18 file in total

    def pagelinks_2nd_step():
        """
        Extracts hyperlink relationships (from_id to target_id) from the chunks.
        
        Filters data to include only namespace 0 (Main/Article namespace) records.
        """
        DATA_DIR = os.path.join(BASE_DIR, 'file', 'pagelinks_chunks')
        # Iterate through the 18 generated chunks
        for nchunk in tqdm(range(1, 19)):
            from_target = {} # Key: from_page_id, Value: List of target_link_ids
            # Define chunk path using os.path.join
            pkl_file_path = os.path.join(CHUNKS_DIR, f"chunk_{nchunk}.pickle")
            with open(pkl_file_path, 'rb') as fr:
                sql_lines = pickle.load(fr)
                
            check_ns = True
            a = 0
            for sql_line in sql_lines:
                # Regex to extract data within parentheses
                a += 1
                pattern = re.compile(r"\((.*?)\)")
                extracted_data = pattern.findall(sql_line)
                processed_data = [tuple(item.split(",")) for item in extracted_data]

                for data in processed_data:
                    # Filter for Namespace 0 (data[1] == '0')
                    if data[1] == '0':
                        pl_from = data[0]
                        pl_target_id = data[2]
                        from_target.setdefault(pl_from, []).append(pl_target_id)
                    else:
                        check_ns = False

            # Save the processed mapping for the current chunk
            chunk_output_file_path = os.path.join(CHUNKS_DIR, f"from_target_chunk_{nchunk}.pickle")
            with open(chunk_output_file_path, "wb") as pkl_file:
                pickle.dump(from_target, pkl_file)
            # Print namespace check result for each chunk
            # Note: Later chunks may return False as namespace changes
            print(f"Chunk {nchunk} Namespace Check: {check_ns}")
        return

    def pagelinks_3rd_step():
        """
        Merges all chunk-specific 'from_target' dictionaries into one final file.
        """
        from_target = {}
        for nchunk in range(1, 19):
            chunk_input_file_path = os.path.join(CHUNKS_DIR, f"from_target_chunk_{nchunk}.pickle")
            with open(chunk_input_file_path, "rb") as pkl_file:
                chunk_from_target = pickle.load(pkl_file)
            
            from_list = list(chunk_from_target.keys())
            for from_id in tqdm(from_list, desc=f"Merging Chunk {nchunk}", leave=False):
                from_target.setdefault(from_id, []).extend(chunk_from_target[from_id])
        
        # [CORRECTION] Saving the final merged dataset to TARGET_DIR
        from_target_file_path = os.path.join(TARGET_DIR, "from_target.pickle")
        with open(from_target_file_path, "wb") as pkl_file:
            pickle.dump(from_target, pkl_file)
            
        print(f"[FINISH] Final dataset saved to: {from_target_file_path}")
        return
    # Execute the steps sequentially
    pagelinks_1st_step()
    pagelinks_2nd_step()
    pagelinks_3rd_step()

    return

In [ ]:
import os

# Be careful because the capacity is large! 32G --> 33G --> 7.7G
Parsing_pagelinks(BASE_DIR)

#### 3.0 MariaDB Access and Management

This section describes the standard procedures for managing the MariaDB service and interacting with the database via the terminal.

1. Check MariaDB Service Status
Verify whether the MariaDB service is currently active or inactive.

```bash
sudo systemctl status mariadb  # Check the current status (Active/Inactive)
```

2. Start MariaDB Service
If the status is reported as inactive (dead), use the following command to start the server:

```Bash
sudo systemctl start mariadb   # Start the MariaDB server process
```

3. Database Connection and Table Inspection
3-1. Connect to the Server
Log in to the MariaDB prompt as the root user (password required).

```Bash
mysql -u root -p               # Access the MariaDB prompt
# The prompt will change to: MariaDB [(none)]>
```

3-2. List Databases
View all databases existing on the server.

```SQL
SHOW DATABASES;                # Display the list of all databases
```

3-3. Select a Specific Database
Replace mydatabase with the name of your target database.

```SQL
USE mydatabase;                # Select the target database
# The prompt will change to: MariaDB [mydatabase]>
```

3-4. List Tables
Check the list of tables within the selected database.

```SQL
SHOW TABLES;                   # Display tables in the current database
```

3-5. Inspect Table Schema
Check the column names, types, and constraints of a specific table (e.g., users).

```SQL
DESCRIBE users;                # View the structure of the 'users' table
# Alternative: DESC users;
```

4. Termination and Service Shutdown
4-1. Exit MariaDB Prompt
Terminate the current session and return to the standard terminal.

```SQL
EXIT;                          # Close the MariaDB prompt
```

4-2. Stop MariaDB Service
Use this command only when you need to completely stop the server process.

```Bash
sudo systemctl stop mariadb    # Stop the running MariaDB server process
```

#### 3. Extract Linktarget Table (MariaDB to Dictionary)

This step converts the linktarget table from MariaDB into a Python dictionary. In the Wikipedia database schema, most hyperlinks do not point directly to a Page ID; instead, they reference a linktarget ID, which then maps to the actual page title.

Description: Extracts mapping between lt_id and lt_title to resolve indirect hyperlink references.

Input: linktarget table stored in MariaDB.

Processing Logic: * Uses a Server-Side Cursor (SSCursor) to handle large-scale data without memory overflow.

Filters for records where lt_namespace is 0 (Main/Article namespace).

Output: Dataset/raw_page2page/linktarget_dict.pickle

Data Structure:

Format: linktarget_dict[lt_id] = lt_title

Data Types: Key: int (lt_id), Value: str (lt_title)

In [ ]:
import pymysql
import pickle
import os
from tqdm.auto import tqdm

def Linktarget_to_dict(BASE_DIR,user, password):
    """
    Retrieves data from the MariaDB 'linktarget' table and saves it as a dictionary in a pickle file.
    
    This function uses SSCursor (Server-Side Cursor) to handle large datasets efficiently 
    by fetching rows in batches and filtering for namespace 0.

    Args:
        BASE_DIR (str): The root directory of the project.
        user (str): MariaDB username.
        password (str): MariaDB password.

    Returns:
        dict: A dictionary mapping lt_id to lt_title for namespace 0.
    """
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_page2page')
    
    # MariaDB connection configuration
    db_config = {
        "host": "localhost",
        "user": user,
        "password": password,
        "database": "enwiki",
        "port": 3306,
        "cursorclass": pymysql.cursors.SSCursor  # Optimized for large datasets
    }

    print("[INFO] Fetching data from 'linktarget' table (Safe Mode for large data)")
    connection = pymysql.connect(**db_config)
    cursor = connection.cursor()

    # 1. Verify total row count for validation
    cursor.execute("SELECT COUNT(*) FROM linktarget")
    db_row_count = cursor.fetchone()[0]
    print(f"[INFO] Total rows in 'linktarget' table: {db_row_count:,}")

    # 2. Fetch data in chunks
    query = "SELECT lt_id, lt_namespace, lt_title FROM linktarget"
    cursor.execute(query)

    linktarget_dict = {}
    batch_size = 100_000
    total_rows = 0

    with tqdm(desc="Processing rows", unit="rows") as pbar:
        while True:
            rows = cursor.fetchmany(batch_size)
            if not rows:
                break
            for lt_id, lt_namespace, lt_title in rows:
                if lt_namespace == 0:
                    linktarget_dict[lt_id] = lt_title
            total_rows += len(rows)
            pbar.update(len(rows))

    cursor.close()
    connection.close()

    print(f"[SUCCESS] Dictionary conversion complete. Total processed: {total_rows:,}")

    # 3. Data Validation (Compare DB rows vs. Processed rows)
    if total_rows == db_row_count:
        print("[SUCCESS] Validation complete: DB and Dictionary row counts match.")
    else:
        print("[ERROR] Data mismatch detected. Some records might be missing.")
        print(f"    - DB Total Rows: {db_row_count:,}")
        print(f"    - Processed Rows: {total_rows:,}")
        print("[WARNING] Please check the database or re-run the process.")

    # 4. Save to Pickle
    print("[INFO] Saving to pickle file...")
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    # Defining file_path using the 'xxx_file_path' format
    linktarget_file_path = os.path.join(TARGET_DIR, 'linktarget_dict.pickle')
    with open(linktarget_file_path, 'wb') as f:
        pickle.dump(linktarget_dict, f)

    print(f"[FINISH] Process complete. {len(linktarget_dict):,} lt_ids saved in: {linktarget_file_path}")
    return linktarget_dict

In [ ]:
import os

Linktarget_to_dict(BASE_DIR, "your_ID", "your_password")

#### 4. Extract Page Table (MariaDB to Dictionary)

This step processes the page table from MariaDB to create bidirectional mapping dictionaries between Page IDs and Titles. These dictionaries are essential for resolving page metadata and identifying redirect status efficiently.

Description: Extracts comprehensive page metadata, including IDs, titles, and redirect flags.

Input: page table stored in MariaDB.

Processing Logic: * Implements batch fetching to manage large-scale data retrieval.

Filters for records where page_namespace is 0 (Main/Article namespace).

Outputs: 1. Dataset/raw_page2page/page_id_dict.pickle
* Maps Page ID to its corresponding Title and Redirect status.
* Format: page_id_dict[page_id] = (page_title, page_is_redirect)
2. Dataset/raw_page2page/page_title_dict.pickle
* Maps Page Title to its corresponding ID and Redirect status.
* Format: page_title_dict[page_title] = (page_id, page_is_redirect)

Data Types: page_id: int, page_title: str, page_is_redirect: int (0 or 1)

In [ ]:
import pymysql
import pickle
from tqdm.auto import tqdm
import os
def Page_to_dict(BASE_DIR, user, password):
    """
    Retrieves data from the MariaDB 'page' table and saves it as dictionary pickle files.
    
    This function extracts page_id, title, namespace, and redirect status, 
    organizing them into mapping dictionaries for efficient data retrieval.

    Args:
        BASE_DIR (str): The root directory of the project.
        user (str): MariaDB username.
        password (str): MariaDB password.

    Returns:
        None: Saves 'page_id_dict.pickle' and 'page_title_dict.pickle' to the target directory.
    """
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_page2page')
    
    # MariaDB connection configuration
    db_config = {
        "host": "localhost",
        "user": user,
        "password": password,
        "database": "enwiki",
        "port": 3306,
        "cursorclass": pymysql.cursors.SSCursor # Optimized for large-scale data
    }

    connection = pymysql.connect(**db_config)
    cursor = connection.cursor()

    # 1. Verify the total row count in the 'page' table
    print("[INFO] Checking row count in 'page' table...")
    cursor.execute("SELECT COUNT(*) FROM page")
    db_row_count = cursor.fetchone()[0]
    print(f"[INFO] Total rows in 'page' table: {db_row_count:,}")

    # 2. Fetch data using a server-side cursor for memory efficiency
    print("[INFO] Fetching data from 'page' table (Large-scale Safe Mode)...")
    cursor.execute("""
        SELECT page_id, page_title, page_namespace, page_is_redirect
        FROM page
    """)
    
    # Initialize dictionaries
    page_title_dict = {}
    ns_list = [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 100, 101, 118, 119, 126, 
               127, 710, 711, 828, 829, 1728, 1729]
    for ns in ns_list:
        page_title_dict[ns]={}

    page_id_dict = {}
    batch_size = 100_000
    total_rows = 0

    # 3. Process rows in batches and filter for namespace 0
    with tqdm(desc="Processing rows (page)", unit="rows") as pbar:
        while True:
            rows = cursor.fetchmany(batch_size)
            if not rows:
                break
            for page_id, page_title, page_namespace, page_is_redirect in rows:
                if page_namespace == 0:
                    page_id_dict[page_id] = (page_title, page_is_redirect)
                    # Note: Existing logic maps title directly to dict if namespace is 0
                    page_title_dict[page_title] = (page_id, page_is_redirect)
            total_rows += len(rows)
            pbar.update(len(rows))

    cursor.close()
    connection.close()

    print(f"[SUCCESS] Data conversion to dictionary complete. Total rows: {total_rows:,}")

    # 4. Validation: Compare DB row count with processed row count
    if total_rows == db_row_count:
        print("[SUCCESS] Validation passed: DB and processed row counts match.")
    else:
        print("[ERROR] Validation failed: Mismatch between DB rows and processed rows.")
        print(f"    - DB Rows: {db_row_count:,} / Processed Rows: {total_rows:,}")

    # 5. Save dictionaries to pickle files
    print("[INFO] Saving data to pickle files...")
    if not os.path.exists(TARGET_DIR):
        os.makedirs(TARGET_DIR, exist_ok=True)
    
    # Defining file paths using the 'xxx_file_path' format
    page_id_file_path = os.path.join(TARGET_DIR, 'page_id_dict.pickle')
    page_title_file_path = os.path.join(TARGET_DIR, 'page_title_dict.pickle')

    with open(page_id_file_path, 'wb') as f:
        pickle.dump(page_id_dict, f)
    with open(page_title_file_path, 'wb') as f:
        pickle.dump(page_title_dict, f)

    print("[FINISH] Process complete. Files saved:")
    print(f"    - page_id_dict: {len(page_id_dict):,} records")
    
    # Calculate total keys across namespaces in page_title_dict for verification
    check = 0
    for ns in ns_list:
        check += len(page_title_dict[ns].keys())
    print(f"    - page_title_dict (per namespace keys): {check:,} records")

    return

In [ ]:
import os

Page_to_dict(BASE_DIR, "your_ID", "your_password")

#### 5. Extract Redirect Table (MariaDB to Dictionary)

This step processes the redirect table from MariaDB to map redirecting pages to their respective target titles. In Wikipedia's structure, a redirect page points to a target link rather than directly to another Page ID.

Description: Extracts redirection data to identify which pages automatically lead to a target article.

Input: redirect table stored in MariaDB.

Processing Logic:

Utilizes SSCursor for memory-efficient data streaming.

Filters for records where rd_namespace is 0 (Main/Article namespace).

Output: Dataset/raw_page2page/redirect_dict.pickle

Data Structure:

Format: redirect_dict[rd_from] = rd_title

Data Types: Key: int (rd_from / Source Page ID), Value: str (rd_title / Target Page Title)

In [ ]:
import pymysql
import pickle
from tqdm.auto import tqdm
import os

def Redirect_to_dict(BASE_DIR, user, password):
    """
    Retrieves data from the MariaDB 'redirect' table and saves it as a dictionary in a pickle file.
    
    This function extracts redirection mapping (rd_from to rd_title) for namespace 0,
    enabling the resolution of redirect pages to their target article titles.

    Args:
        BASE_DIR (str): The root directory of the project.
        user (str): MariaDB username.
        password (str): MariaDB password.

    Returns:
        None: Saves 'redirect_dict.pickle' to the target directory.
    """
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_page2page')
    
    # 1. MariaDB Connection Configuration
    db_config = {
        "host": "localhost",
        "user": user,
        "password": password,
        "database": "enwiki",
        "port": 3306,
        "cursorclass": pymysql.cursors.SSCursor  # Optimized for large-scale data retrieval
    }

    # 2. Establish Connection and Verify Row Count
    connection = pymysql.connect(**db_config)
    cursor = connection.cursor()

    print("[INFO] Checking row count in 'redirect' table...")
    cursor.execute("SELECT COUNT(*) FROM redirect")
    db_row_count = cursor.fetchone()[0]
    print(f"[INFO] Total rows in 'redirect' table: {db_row_count:,}")

    # 3. Fetch Data from Redirect Table
    print("[INFO] Fetching data from 'redirect' table (Large-scale Safe Mode)...")
    cursor.execute("""
        SELECT rd_from, rd_namespace, rd_title
        FROM redirect
    """)

    redirect_dict = {}
    batch_size = 100_000
    total_rows = 0

    with tqdm(desc="Processing rows (redirect)", unit="rows") as pbar:
        while True:
            rows = cursor.fetchmany(batch_size)
            if not rows:
                break
            for rd_from, rd_namespace, rd_title in rows:
                if rd_namespace == 0:
                    redirect_dict[rd_from] = rd_title
            total_rows += len(rows)
            pbar.update(len(rows))

    cursor.close()
    connection.close()

    print(f"[SUCCESS] Data conversion to dictionary complete. Total processed: {total_rows:,}")

    # 4. Validation
    if total_rows == db_row_count:
        print("[SUCCESS] Validation complete: DB and dictionary row counts match.")
    else:
        print("[ERROR] Data mismatch detected. Some rows might be missing.")
        print(f"    - DB Total Rows: {db_row_count:,}")
        print(f"    - Processed Rows: {total_rows:,}")
        print("[WARNING] Please re-run the process or check the data status.")

    # 5. Save to Pickle
    print("[INFO] Saving to pickle file...")
    if not os.path.exists(TARGET_DIR):
        os.makedirs(TARGET_DIR, exist_ok=True)
    
    # Defining file_path using the 'xxx_file_path' format
    redirect_file_path = os.path.join(TARGET_DIR, 'redirect_dict.pickle')
    with open(redirect_file_path, 'wb') as f:
        pickle.dump(redirect_dict, f)

    print(f"[FINISH] Process complete. A total of {len(redirect_dict):,} 'rd_from' entries saved in: {redirect_file_path}")
    return


In [ ]:
import os

Redirect_to_dict(BASE_DIR, "your_ID", "your_password")

## Step 1-2: Extract data

### Step 1-2-1: Hyperlink dataset

#### 1. Generate Page-to-Page Mapping

This step constructs a comprehensive hyperlink mapping between Wikipedia articles. It processes raw SQL-derived data to build a directed graph where each source page points to a list of its target pages, ensuring all redirects are resolved to their final destinations.

Description: Extracts and cleans page-to-page relationships.

Resolves linktarget IDs to actual Page IDs.

Recursively follows redirection chains (up to 100,000 steps) to ensure the target is a valid, non-redirecting article (ns=0).

Note: Multiple links to the same target may exist, representing edge weights (if applicable in further analysis).

Input: All pre-processed pickle files (from_target, linktarget_dict, page_id_dict, page_title_dict, redirect_dict).

Output: Dataset/raw_page2page/page2page.pickle

Data Structure:

Format: page2page[page1_id] = [page2_id, page2_id, ...]

Data Types: Key: str (Source Page ID), Value: List[int] (Target Page IDs)

In [ ]:
import os
import pickle
from tqdm.auto import tqdm

def Make_page2page(BASE_DIR):
    """
    Constructs a page-to-page hyperlink mapping by resolving linktarget IDs and handling redirects.

    This function loads pre-processed dictionaries, iterates through article links, 
    and follows redirection chains until a final article (namespace 0) is reached.

    Args:
        BASE_DIR (str): The root directory of the project.

    Returns:
        None: Saves the final 'page2page.pickle' to the target directory.
    """
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_page2page')
    TARGET_DIR = DATA_DIR
    
    def load_files():
        """Loads required pickle dictionaries for the page2page mapping process."""
        # Construct absolute file paths for each required resource
        from_target_file_path = os.path.join(BASE_DIR, "file", "from_target.pickle")
        linktarget_file_path = os.path.join(DATA_DIR, "linktarget_dict.pickle")
        page_title_file_path = os.path.join(DATA_DIR, "page_title_dict.pickle")
        page_id_file_path = os.path.join(DATA_DIR, "page_id_dict.pickle")
        redirect_file_path = os.path.join(DATA_DIR, "redirect_dict.pickle")

        # Loading datasets from storage
        with open(from_target_file_path, "rb") as pkl_file:
            from_target = pickle.load(pkl_file)

        with open(linktarget_file_path, "rb") as pkl_file:
            linktarget_dict = pickle.load(pkl_file)

        with open(page_title_file_path, "rb") as pkl_file:
            page_title_dict = pickle.load(pkl_file)

        with open(page_id_file_path, "rb") as pkl_file:
            page_id_dict = pickle.load(pkl_file)

        with open(redirect_file_path, "rb") as pkl_file:
            redirect_dict = pickle.load(pkl_file)
            
        return from_target, linktarget_dict, page_title_dict, page_id_dict, redirect_dict
    # Unpack loaded datasets
    from_target, linktarget_dict, page_title_dict, page_id_dict, redirect_dict = load_files()

    page1_list = list(from_target.keys())
    page2page = {}
    
    for page1 in tqdm(page1_list, desc="Building page-to-page mapping"):
        try:
            # Check if page1 exists in the ID dictionary
            page1_title, page1_is_redirect = page_id_dict[int(page1)]
            # Process only if the source page is not a redirect page
            # (Users typically navigate from original article content to target links)
            if page1_is_redirect == 0:
                page2_list = []
                page2_lt_id_list = from_target[page1] # List of linktarget IDs (string format)
                
                for page2_lt_id in page2_lt_id_list:
                    try:
                        # Convert linktarget ID to page title
                        page2_title = linktarget_dict[int(page2_lt_id)] # page2_ns = int
                        # Get ID and redirect status for the target page title
                        page2_id, page2_is_redirect = page_title_dict[page2_title]
                        # If the target is a redirect, follow the chain until reaching a non-redirect page
                        if page2_is_redirect == 1:
                            page2_redi_check = 0
                            while page2_redi_check < 100000:
                                page2_final_title = redirect_dict[page2_id]
                                page2_redi_id, page2_is_redirect = page_title_dict[page2_final_title]
                                if page2_is_redirect == 0:
                                    page2_final_id = page2_redi_id
                                    break
                                else:
                                    page2_id = page2_redi_id
                                page2_redi_check += 1
                        else:
                            page2_final_id = page2_id
                        page2_list.append(page2_final_id)
                    except KeyError:
                        # Skip if the linktarget or title is not found
                        pass
                if page2_list:
                    page2page.setdefault(page1, []).extend(page2_list)
        except KeyError as e:
            # Skip if source page ID is not found in the dictionary
            pass

    # Save the final page-to-page hyperlink dataset
    page2page_file_path = os.path.join(TARGET_DIR, "page2page.pickle")
    with open(page2page_file_path, "wb") as pkl_file:
        pickle.dump(page2page, pkl_file)
        
    print(f"[FINISH] Mapping process complete. Output saved to: {page2page_file_path}")
    return

In [ ]:
Make_page2page(BASE_DIR)

#### 2-1. Degree Distribution Analysis

In [ ]:
import os
import pickle
from collections import Counter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Define directory paths for data access
DATA_DIR = os.path.join(BASE_DIR, "Dataset", "raw_page2page")

# Construct the absolute file path for the input dataset
page2page_file_path = os.path.join(DATA_DIR, "page2page.pickle")

# Load the serialized page-to-page hyperlink mapping data
with open(page2page_file_path, "rb") as pkl_file:
    page2page = pickle.load(pkl_file)

# Compute the unique out-degree for each source page
# Utilizing set() ensures that only unique target links are counted per page
degrees = [len(set(page2page[page1])) for page1 in tqdm(page2page.keys(), desc="Calculating Out-Degrees")]
degree_counts = Counter(degrees)

# Initialize the plot for degree distribution analysis
plt.figure(figsize=(8, 5))

# Generate a log-log scatter plot to visualize the network's degree distribution
plt.scatter(degree_counts.keys(), degree_counts.values(), alpha=0.6)

# Apply logarithmic scales to both axes to identify potential power-law behavior
plt.xscale('log')
plt.yscale('log')

# Set academic labels and titles for the visualization
plt.xlabel('Degree (Unique links)')
plt.ylabel('Count (Number of pages)')
plt.title('Page-to-Page Degree Distribution')

# Enhance readability with a subtle grid structure
plt.grid(True, which="both", ls="-", alpha=0.2)

# Display the finalized distribution plot
plt.show()

#### 2-2. Edge Weight Distribution Analysis

In [ ]:
import os
import pickle
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Define directory and file paths for consistency
# DATA_DIR is defined to manage the dataset location
DATA_DIR = os.path.join(BASE_DIR, "Dataset", "raw_page2page")
page2page_file_path = os.path.join(DATA_DIR, "page2page.pickle")

# Calculate edge weights representing the frequency of repeated links between page pairs
weights_list = []
page1_ids = list(page2page.keys())

# Iterate through source pages to determine link frequency for each target
for page1 in tqdm(page1_ids, desc="Calculating Edge Weights"):
    page2_ids = page2page[page1]
    link_counts = {}
    
    # Quantify occurrences of each unique target page per source page
    for page2 in page2_ids:
        link_counts[page2] = link_counts.get(page2, 0) + 1
    
    # Aggregate all weight values into a collective list
    weights_list.extend(link_counts.values())

# Compute the global frequency distribution of identified weight values
weight_distribution = {}
for weight in weights_list:
    weight_distribution[weight] = weight_distribution.get(weight, 0) + 1

# Initialize visual analysis of the weight distribution
plt.figure(figsize=(8, 5))

# Generate a scatter plot using a log-log scale to observe link frequency patterns
plt.scatter(weight_distribution.keys(), weight_distribution.values(), color='orange', alpha=0.6)

# Configure logarithmic scaling for both axes to represent heavy-tailed data
plt.xscale('log')
plt.yscale('log')

# Apply standardized labels and title for academic reporting
plt.xlabel('Weight (Link frequency)')
plt.ylabel('Count (Number of edges)')
plt.title('Page-to-Page Weight Distribution')

# Add a subtle grid to facilitate quantitative estimation from the plot
plt.grid(True, which="both", ls="-", alpha=0.2)

# Render the finalized distribution plot
plt.show()

#### 3. Generate Unweighted Dataset (wow_page2page

This step removes redundant links between the same pair of pages to create an unweighted version of the hyperlink network (WithOut Weight).

Description: Converts the weighted adjacency list into an unweighted format by removing duplicate target IDs for each source page.

Input: Dataset/raw_page2page/page2page.pickle

Output: Dataset/wow_page2page.pickle

Data Structure: wow_page2page[page1_id] = [unique_page2_id, ...]

In [ ]:
import os
import pickle
from tqdm.auto import tqdm

def Without_weight_page2page(BASE_DIR):
    """
    Removes redundant edges from the weighted page2page mapping to generate an unweighted dataset.
    This ensures that each hyperlink between two unique pages is represented only once.
    """
    
    # Define standardized directory paths
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_page2page')
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    # Construct input file path using standardized naming convention
    page2page_file_path = os.path.join(DATA_DIR, 'page2page.pickle')
    
    # Load the serialized weighted hyperlink mapping
    with open(page2page_file_path, 'rb') as fr:
        page2page = pickle.load(fr)

    wow_page2page = {}
    source_page_ids = list(page2page.keys())
    
    # Iterate through each source page to filter duplicate target links
    for page1 in tqdm(source_page_ids, desc="Generating Unweighted Network"):
        target_page_ids = page2page[page1]
        
        # Deduplicate target IDs by converting the list to a set and back to a list
        wow_page2page[page1] = list(set(target_page_ids))

    # Define the final output file path in the main Dataset directory
    wow_page2page_file_path = os.path.join(TARGET_DIR, 'wow_page2page.pickle')
    
    # Serialize and save the unweighted (WOW) dataset
    with open(wow_page2page_file_path, 'wb') as fw:
        pickle.dump(wow_page2page, fw)
        
    print(f"[FINISH] Unweighted dataset (WOW) saved to: {wow_page2page_file_path}")
    return

In [ ]:
Without_weight_page2page(BASE_DIR)

#### 4-1. Unweighted Degree Distribution Analysis

In [ ]:
import os
import pickle
from collections import Counter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Define the data directory for internal data management
DATA_DIR = os.path.join(BASE_DIR, "Dataset")

# Construct the absolute file path for the unweighted dataset
wow_page2page_file_path = os.path.join(DATA_DIR, "wow_page2page.pickle")

# Load the unweighted (WOW) page-to-page mapping data from the serialized pickle file
with open(wow_page2page_file_path, "rb") as pkl_file:
    wow_page2page = pickle.load(pkl_file)

# Calculate the out-degree for each unique source page
# Since the WOW dataset is pre-deduplicated, the length of each list represents the unique degree
degrees = [len(wow_page2page[page1]) for page1 in tqdm(wow_page2page.keys(), desc="Calculating Degrees")]
degree_counts = Counter(degrees)

# Initialize the visualization for the unweighted degree distribution
plt.figure(figsize=(8, 5))

# Generate a scatter plot on a log-log scale to characterize the network topology
plt.scatter(degree_counts.keys(), degree_counts.values(), alpha=0.6, color='blue')

# Apply logarithmic scaling to both axes to evaluate power-law characteristics
plt.xscale('log')
plt.yscale('log')

# Define academic labels and title for the resulting distribution plot
plt.xlabel('Degree (Unique links)')
plt.ylabel('Count (Number of pages)')
plt.title('Unweighted Page-to-Page Degree Distribution (WOW)')

# Incorporate a subtle grid to aid in quantitative interpretation
plt.grid(True, which="both", ls="-", alpha=0.2)

# Render the finalized visualization
plt.show()

#### 4-2. Unweighted Weight Distribution Analysis

In [ ]:
import os
import pickle
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Define the data directory for internal resource management
DATA_DIR = os.path.join(BASE_DIR, "Dataset")

# Construct the absolute file path for the unweighted dataset to be verified
wow_page2page_file_path = os.path.join(DATA_DIR, "wow_page2page.pickle")

# Load the unweighted (WOW) page-to-page mapping data
with open(wow_page2page_file_path, "rb") as pkl_file:
    wow_page2page = pickle.load(pkl_file)

# Aggregate link weights from the unweighted dataset to verify edge consistency
edge_weights = []
source_page_ids = list(wow_page2page.keys())

# Iterate through source pages to calculate occurrences of each unique target link
for page1 in tqdm(source_page_ids, desc="Verifying Edge Weights"):
    target_pages = wow_page2page[page1]
    link_counts = {}
    
    # Quantify the frequency of each target link per source page
    for page2 in target_pages:
        link_counts[page2] = link_counts.get(page2, 0) + 1
    
    # Append calculated weights to the global list for distribution analysis
    edge_weights.extend(link_counts.values())

# Compute the frequency distribution of observed edge weights
weight_distribution = {}
for w in edge_weights:
    weight_distribution[w] = weight_distribution.get(w, 0) + 1

# Initialize the plot to visualize the weight distribution of the WOW dataset
# Note: For an unweighted network, all observed weights must equal 1
plt.figure(figsize=(8, 5))

# Generate a scatter plot on a log-log scale to identify potential weight anomalies
plt.scatter(weight_distribution.keys(), weight_distribution.values(), color='red', alpha=0.6)

# Configure logarithmic scales for axes consistency across dataset comparisons
plt.xscale('log')
plt.yscale('log')

# Set standardized labels and titles for academic reporting
plt.xlabel('Weight (Link frequency)')
plt.ylabel('Count (Number of edges)')
plt.title('Unweighted Page-to-Page Weight Distribution (WOW)')

# Apply a subtle grid to assist in the visual verification of data points
plt.grid(True, which="both", ls="-", alpha=0.2)

# Display the finalized verification plot
plt.show()

# Final integrity check to confirm all edge weights are exactly 1
if all(w == 1 for w in weight_distribution.keys()):
    print("[SUCCESS] Verification complete: All edge weights in the WOW dataset are 1.")
else:
    print("[WARNING] Unexpected weights found. Please verify the 'Without_weight_page2page' function.")

### Step 1-2-2: Page history dataset

#### 1. Indexing Network Pages in History (page_info_loc)

This step identifies which pages from the wow_page2page network exist in the Wikipedia history dumps and records their specific storage locations and event counts.

Description: Creates a lookup table to quickly locate a page's history across multiple partitioned files.

Input: Dataset/wow_page2page.pickle, Dataset/raw_history/history_ns0_{nloc}.pickle

Processing Logic: * Collects all unique page IDs from the network (source and target).

Iterates through 27 history chunks to map each page to its corresponding nloc (file index) and Nevents (edit count).

Outputs: 1. Dataset/page_info_loc.pickle
* Format: page_info_loc[page_id] = [Nevents, nloc]
2. Dataset/raw_page_edit_counts.pickle
* Contains raw edit counts for all pages in the history.

In [ ]:
import pickle
import os
from tqdm.auto import tqdm

def Page_info_loc(BASE_DIR):
    """
    Identifies pages present in the page2page network and indexes their history location.
    
    This function filters history data to include only pages involved in the network,
    recording the number of events and the source file index (nloc).
    """
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    
    # Construct input file path for the unweighted network
    wow_page2page_file_path = os.path.join(DATA_DIR, "wow_page2page.pickle")
    
    print(f"[INFO] Loading network data from: {wow_page2page_file_path}")
    with open(wow_page2page_file_path, "rb") as pkl_file:
        page2page = pickle.load(pkl_file)
        
    # Collect all unique pages (source and target) in the network
    check = 0
    page_list = []
    for page1 in page2page.keys():
        page_list.append(page1)
        page_list.extend(page2page[page1])
        # Periodic duplicate removal to manage memory
        if check % 1000000 == 0:
            page_list = list(set(page_list))
        check += 1

    page_list = list(set(page_list))
    print(f"[INFO] Total unique pages in network: {len(page_list)}")
    
    # Release memory for the original network dictionary
    page2page = 0
    page_dict = dict.fromkeys(page_list, 0)
    
    page_info_loc = {}
    count_final_events = 0
    count_page_history = {}
    count_Nevents = 0
    
    # Iterate through 27 raw history chunks
    for nloc in tqdm(range(1, 28), desc="Indexing history files"):
        file_path = os.path.join(DATA_DIR, 'raw_history', f'history_ns0_{nloc}.pickle')
        with open(file_path, 'rb') as fr:
            histories = pickle.load(fr)

        pages = list(histories.keys())
        for page in pages:
            # histories[page][1] contains the list of events
            Nevents = len(histories[page][1])
            count_page_history[page] = Nevents
            count_Nevents += Nevents
            
            # Record location if the page is part of the network
            if page in page_dict:
                page_info_loc[page] = [Nevents, nloc]
                count_final_events += Nevents
            
    print(f"[RESULTS] Total history pages: {len(count_page_history.keys())}")
    print(f"[RESULTS] Total history edits: {count_Nevents}")
    print(f"[RESULTS] Filtered network pages: {len(page_info_loc.keys())}")
    print(f"[RESULTS] Filtered network edits: {count_final_events}")
    
    # Save the indexing results
    page_info_loc_file_path = os.path.join(DATA_DIR, 'page_info_loc.pickle')
    with open(page_info_loc_file_path, 'wb') as fw:
        pickle.dump(page_info_loc, fw)
        
    raw_edit_counts_file_path = os.path.join(DATA_DIR, 'raw_page_edit_counts.pickle')
    with open(raw_edit_counts_file_path, 'wb') as fw:
        pickle.dump(raw_page_edit_counts, fw)
        
    print(f"[FINISH] Metadata indexing complete. Results saved to {DATA_DIR}")
        
    return

In [ ]:
import os

Page_info_loc(BASE_DIR)

#### 2. Extract Conditioned History and Pairs (Used_edit_pair_history)

This step filters the dataset to retain only highly active pages (e.g., those with more than 1,000 edits) and their corresponding network connections.

Description: Extracts a sub-network and its revision history based on a minimum edit threshold (edit_number).

Input: Dataset/wow_page2page.pickle, Dataset/page_info_loc.pickle

Processing Logic:

Filters pairs (page1, page2) where both pages meet the edit_number criteria.

Aggregates the full revision history for these filtered pages into a single optimized file.

Outputs:

Dataset/conditioned_history/condi_hist_{edit_number}.pickle

Contains full revision history for filtered pages.

Format: condition_history[page_id] = [page_title, [events...]]

Dataset/conditioned_history/condi_pairs_{edit_number}.pickle

A list of validated pairs meeting the activity threshold.

Format: pair_list = [(page1, page2), ...]

In [ ]:
import pickle
import os
from collections import defaultdict
from tqdm.auto import tqdm

def Used_edit_pair_history(BASE_DIR, edit_number):
    """
    Extracts history and pairs for pages that meet the minimum edit count criteria.
    
    Args:
        BASE_DIR (str): Root directory of the project.
        edit_number (int): Minimum number of edits required for filtering.
    """
    # Define standardized directory paths
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    TARGET_DIR = os.path.join(DATA_DIR, 'conditioned_history')
    
    # Ensure the output directory for conditioned data exists
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """Loads required indexing and network dictionaries using standardized path naming."""
        # Define absolute file paths for input resources
        page2page_file_path = os.path.join(DATA_DIR, 'wow_page2page.pickle')
        page_info_loc_file_path = os.path.join(DATA_DIR, 'page_info_loc.pickle')
        
        with open(page2page_file_path, 'rb') as fr:
            page2page = pickle.load(fr)

        with open(page_info_loc_file_path, 'rb') as fr:
            page_info_loc = pickle.load(fr)
            
        return page_info_loc, page2page

    page_info_loc, page2page = load_files()
    print("[INFO] Files loaded successfully.")
    
    def find_ToF(page, edit_number):
        """Checks if a page meets the edit count threshold."""
        try:
            N_events = page_info_loc[page][0]
            if N_events < edit_number:
                ToF = False
            else:
                ToF = True
        except KeyError:
            ToF = False
        return ToF

    pair_list = []
    # Map each location to a list of pages to be extracted
    pages = {nloc: [] for nloc in range(1, 28)}

    for page1 in tqdm(page_info_loc.keys(), desc="Filtering pairs"):
        page1_ToF = find_ToF(page1, edit_number)
        if page1_ToF == True:
            page1_loc = page_info_loc[page1][1]
            page2s = page2page[page1]
            for page2 in page2s:
                page2 = str(page2)
                page2_ToF = find_ToF(page2, edit_number)
                if page2_ToF == True:
                    page2_loc = page_info_loc[page2][1]
                    pair = (str(page1), str(page2))
                    pair_list.append(pair)
                    # Track which pages to extract from which file
                    pages[page1_loc].append(page1)
                    pages[page2_loc].append(page2)
                    
    # Remove duplicates
    pair_list = list(set(pair_list))
    for loc in range(1, 28):
        pages[loc] = list(set(pages[loc]))

    condition_history = {}
    for nloc in tqdm(range(1, 28), desc="Extracting conditioned history"):
        file_path = os.path.join(DATA_DIR, 'raw_history', f'history_ns0_{nloc}.pickle')
        with open(file_path, 'rb') as fr:
            page_history = pickle.load(fr)

        loc_pages = pages[nloc]
        for page in loc_pages:
            condition_history[page] = page_history[page]
    
    print("[INFO] Saving filtered results...")
    # Save the extracted revision histories for pages meeting the threshold
    condi_hist_file_path = os.path.join(TARGET_DIR, f'condi_hist_{edit_number}.pickle')
    with open(condi_hist_file_path, 'wb') as fw:
        pickle.dump(condition_history, fw)
                         
    print(f"[RESULTS] Total pages meeting {edit_number}+ edit threshold: {len(condition_history.keys())}")
    print(f"[RESULTS] Total unique interaction pairs identified: {len(pair_list)}")
                         
    # Save the list of validated interaction pairs
    condi_pairs_file_path = os.path.join(TARGET_DIR, f'condi_pairs_{edit_number}.pickle')
    with open(condi_pairs_file_path, 'wb') as fw:
        pickle.dump(pair_list, fw)
        
    print(f"[FINISH] Data conditioning complete. Output saved to: {TARGET_DIR}")
    return

In [ ]:
Used_edit_pair_history(1000)

### Step 1-2-3: Editor history dataset

#### 1. Extract Editor Histories (Global & Network-based)

This section reconstructs revision histories from an editor-centric perspective by aggregating individual edit events into unified editor profiles.

CRITICAL WARNING: These processes (especially Get_p2p_editors) are extremely memory-intensive. It is highly recommended to monitor system RAM during execution.

1-1. Global Editor Activity (Get_raw_editors)
Scans all history chunks to calculate the total edit frequency for every unique editor.

Input: Dataset/raw_history/history_ns0_{nloc}.pickle

Output: Dataset/raw_editor_history/raw_editor_edit_count.pickle

Format: editor_edits[editor_id_name] = total_count

1-2. Network-based Editor History (Get_p2p_editors)
Extracts and categorizes the complete history of editors who contributed to pages within the wow_page2page network.

Threshold: Editors are categorized by an activity level of 500 edits.

Inputs: wow_page2page.pickle, history_ns0_{nloc}.pickle

Outputs: (Saved in Dataset/raw_editor_history/)

raw_p2p_editor_over_500.pickle: History of editors with edits >= 500.

raw_p2p_editor_under_500.pickle: History of editors with edits < 500.

raw_p2p_editor_edit_count.pickle: Edit counts for network-active editors.

Format: editor_histories[editor_id_name] = [[timestamp, page_id], ...]

In [ ]:
import pickle
import os
from tqdm.auto import tqdm

def Get_raw_editors(BASE_DIR):
    """
    Calculates the total edit counts for each editor across all history chunks.
    
    This function iterates through all namespace 0 history files, aggregates 
    the number of edits per editor, and saves the results as a pickle file.

    Args:
        BASE_DIR (str): The root directory of the project.

    Returns:
        None: Saves 'raw_editor_edit_count.pickle' to the target directory.
    """
    # Define standardized directory paths for input and output
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_editor_history')
    
    # Ensure the destination directory exists
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    # Dictionary to store accumulated edit counts per editor
    editor_edit_counts = {}
    
    # Process each of the 27 history chunks sequentially
    for nloc in tqdm(range(1, 28), desc="Aggregating Editor Activity"):
        history_file_name = f"history_ns0_{nloc}.pickle"
        history_file_path = os.path.join(DATA_DIR, history_file_name)
        
        with open(history_file_path, 'rb') as fr:
            histories = pickle.load(fr)
        
        with open(file_path, 'rb') as fr:
            histories = pickle.load(fr)
        # Iterate through pages in the current history chunk
        for page in tqdm(histories.keys()):
            # histories[page][1] contains the list of revision events
            events = histories[page][1]
            for event in events:
                # event[0]: timestamp, event[1]: editor_id_name
                editor = event[1]
                # Increment the edit count for the editor
                editor_edits[editor] = editor_edits.get(editor, 0) + 1
                
    # Define the final output file path using standardized naming
    editor_count_file_path = os.path.join(TARGET_DIR, 'raw_editor_edit_count.pickle')
    
    # Serialize and save the aggregated editor activity data
    with open(editor_count_file_path, 'wb') as fw:
        pickle.dump(editor_edit_counts, fw)

    print(f"[FINISH] Raw editor activity profile saved to: {editor_count_file_path}")
    return

In [ ]:
import pickle
import os
from tqdm.auto import tqdm

def Get_p2p_editors(BASE_DIR):
    """
    Extracts revision histories for editors within the p2p network.
    
    CRITICAL: This function requires a significant amount of RAM as it loads 
    large-scale dictionaries and processes millions of events simultaneously.
    It is highly recommended to monitor memory usage during execution.
    
    Args:
        BASE_DIR (str): The root directory of the project.
        
    Returns:
        None: Saves categorized editor history and edit counts as pickle files.
    """
    
    # Define standardized directory paths for input and output resources
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Dataset', 'raw_editor_history')
    
    # Ensure the destination directory exists prior to data extraction
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    # Step 1: Identify unique pages involved in the network topology
    print("[WARNING] High Memory Usage Expected: Loading network structure...")
    wow_page2page_file_path = os.path.join(BASE_DIR, "Dataset", "wow_page2page.pickle")
    with open(wow_page2page_file_path, "rb") as pkl_file:
        page2page = pickle.load(pkl_file)
        
    check = 0
    page_list = []
    for page1 in page2page.keys():
        page_list.append(page1)
        page_list.extend(page2page[page1])
        if check % 1000000 == 0:
            page_list = list(set(page_list))
        check += 1
    page_list = list(set(page_list))
    print(f"[INFO] Unique pages in network: {len(page_list)}")
    
    page2page = None
    page_dict = dict.fromkeys(page_list, 0)
    
    editors = {}
    editor_edits = {}
    # Step 2: Iterate through 27 history chunks
    for nloc in tqdm(range(1, 28), desc="Processing history chunks"):
        file_path = os.path.join(DATA_DIR, f'history_ns0_{nloc}.pickle')
        with open(file_path, 'rb') as fr:
            histories = pickle.load(fr)

        for page in tqdm(histories.keys()):
            if page in page_dict:
                page_check = page_dict[page]
                events = histories[page][1]
                for event in events:
                    timestamp = event[0]
                    editor = event[1]
                    editors.setdefault(editor, []).append([timestamp, page])
                    editor_edits[editor] = editor_edits.get(editor, 0) + 1

        # Explicitly clear histories dictionary for every chunk to prevent memory buildup
        histories = None

    # Step 3: Categorize based on edit threshold (500)
    print("[INFO] Categorizing editors (Threshold: 500)...")
    editor_over_500 = {}
    editor_under_500 = {}
    
    for editor, count in tqdm(editor_edits.items(), desc="Categorizing"):
        if count >= 500:
            editor_over_500[editor] = editors[editor]
        else:
            editor_under_500[editor] = editors[editor]
            
    # Free the large 'editors' dictionary before saving
    editors = None
    
    # Step 4: Save datasets
    print("[INFO] Saving finalized datasets using standardized naming...")
    save_tasks = [
        ('raw_p2p_editor_over_500.pickle', editor_over_500_history),
        ('raw_p2p_editor_under_500.pickle', editor_under_500_history),
        ('raw_p2p_editor_edit_count.pickle', editor_edit_counts)
    ]
    
    for file_name, data_object in save_tasks:
        final_output_file_path = os.path.join(TARGET_DIR, file_name)
        with open(final_output_file_path, 'wb') as fw:
            pickle.dump(data_object, fw)
        
        # Explicitly clear the data object after persistence to manage remaining RAM
        data_object = None 
        
    print(f"[FINISH] Editor data extraction complete. System memory should be released.")
    return

In [ ]:
Get_p2p_editors(BASE_DIR)

#### 2. Filter Editor History by Activity Threshold (Used_editors_history)

This step filters the editor dataset to retain only those who meet a specific minimum edit count (e.g., 1,000 edits).

Description: Extracts full revision histories for highly active editors based on the specified edit_number.

Input: * Dataset/raw_editor_history/raw_p2p_editor_edit_count.pickle

Dataset/raw_editor_history/raw_p2p_editor_over_500.pickle

Output: Dataset/conditioned_history/condi_editors_history_{edit_number}.pickle

Format: conditioned_history[editor_id] = [[timestamp, page_id], ...]

In [ ]:
import os
import pickle
from tqdm.auto import tqdm

def Used_editors_history(BASE_DIR, edit_number):
    """
    Filters the editor history dataset based on a minimum edit threshold.

    Args:
        BASE_DIR (str): The root directory of the project.
        edit_number (int): The minimum number of edits required for an editor to be included.

    Returns:
        None: Saves the filtered editor history as a pickle file.
    """
    # Define standardized directory paths for data access
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    # TARGET_DIR is set to the specific folder for conditioned results
    TARGET_DIR = os.path.join(DATA_DIR, 'conditioned_history')
    
    # Ensure the target directory for conditioned results exists
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    # 1. Load the cumulative edit counts for each editor
    # Construct file path using standardized naming convention
    editor_count_file_path = os.path.join(DATA_DIR, 'raw_editor_history', 'raw_p2p_editor_edit_count.pickle')
    with open(editor_count_file_path, 'rb') as fr:
        editor_edits = pickle.load(fr)
        
    conditioned_history = {}
        
    # 2. Identify editors who meet the edit threshold
    editor_list = []
    for editor in editor_edits.keys():
        if editor_edits[editor] >= edit_number:
            editor_list.append(editor)

    # 3. Load pre-categorized histories (editors with 500+ edits)
    # Note: This logic assumes edit_number >= 500
    history_path = os.path.join(DATA_DIR, 'raw_editor_history', 'raw_p2p_editor_over_500.pickle')
    with open(history_path, 'rb') as fr:
        histories = pickle.load(fr)
    
    # 4. Extract history for selected editors
    for editor in editor_list:
        conditioned_history[editor] = histories[editor]

    # 5. Save the filtered dataset
    output_path = os.path.join(TARGET_DIR, f'condi_editors_history_{edit_number}.pickle')
    with open(output_path, 'wb') as fw:
        pickle.dump(conditioned_history, fw)

    print(f"[FINISH] Filtered editor history saved to: {output_path}")
    return

#### 3. Final Dataset Integration and Verification

This step refines the dataset by applying dual activity thresholds for both editors and pages, ensuring high data quality for network analysis.

3-1. Dual-Conditioned Filtering (Used_editors_pages_history)
Filters the dataset to retain only those events where both the editor and the associated pages meet the specified edit count criteria.

Inputs: condi_pairs_{p_num}.pickle, condi_editors_history_{e_num}.pickle

Outputs: (Saved in Dataset/conditioned_history/)

condi_pages_{p_num}_editors_{e_num}.pickle: Final refined history mapping.

condi_pairs_{p_num}_{e_num}.pickle: Final validated network pair list.

Format: history[editor_id] = [[timestamp, page_id], ...]

3-2. Dataset Statistics Verification (check_editors_edits_pages_hyperlinks)
Displays summary statistics for the final integrated dataset to verify the scale of the filtered data.

Metrics: Total count of unique editors, total revision events, unique pages, and identified hyperlinks.

Purpose: Ensures the consistency and scale of the data before proceeding to main analysis.

In [ ]:
import os
import pickle
from tqdm.auto import tqdm

def Used_editors_pages_history(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Refines editor histories and network pairs based on dual thresholds (page and editor).

    Args:
        BASE_DIR (str): The root directory of the project.
        page_edit_number (int): Minimum edit threshold for pages.
        editor_edit_number (int): Minimum edit threshold for editors.

    Returns:
        None: Saves the final integrated history and pair list as pickle files.
    """
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    INPUT_DIR = os.path.join(DATA_DIR, 'conditioned_history')

    # 1. Load initial pair list and editor histories using standardized naming conventions
    pair_file_path = os.path.join(INPUT_DIR, f'condi_pairs_{page_edit_number}.pickle')
    with open(pair_file_path, 'rb') as fr:
        pair_list = pickle.load(fr)
        
    editor_hist_file_path = os.path.join(INPUT_DIR, f'condi_editors_history_{editor_edit_number}.pickle')
    with open(editor_hist_file_path, 'rb') as fr:
        editor_histories = pickle.load(fr)
        
    # 2. Map all pages present in the initial pair list
    page_list = []
    for page1, page2 in pair_list:
        page_list.append(page1)
        page_list.append(page2)
    page_list = list(set(page_list))
    page_dict = dict.fromkeys(page_list, 0)
    
    # 3. Filter history by page presence and sort by timestamp
    final_pages = []
    conditioned_history = {}
    for editor in editor_histories.keys():
        events = editor_histories[editor]
        new_events = []
        for event in events:
            page = event[1]
            try:
                check = page_dict[page]
                new_events.append(event)
                final_pages.append(page)
            except KeyError: pass
        # Sort events by timestamp
        new_events.sort(key=lambda x: x[0])
        
        if new_events != []:
            conditioned_history[editor] = new_events
            
    # Save the final filtered history
    output_hist_name = f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle'
    output_hist_path = os.path.join(INPUT_DIR, output_hist_name)
    with open(output_hist_path, 'wb') as fw:
        pickle.dump(conditioned_history, fw)
    
    # Explicitly clear memory
    conditioned_history = None
    
    # 4. Finalize the pair list based on refined page presence
    final_pages = list(set(final_pages))
    final_page_dict = dict.fromkeys(final_pages, 0)
        
    new_pair_list = []
    for page1, page2 in pair_list:
        try:
            check = final_page_dict[page1]
            try:
                check = final_page_dict[page2]
                new_pair_list.append((page1, page2))
            except KeyError: pass
        except KeyError: pass
    
    # Serialize and save the final refined interaction pair list
    output_pair_file_path = os.path.join(INPUT_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
    with open(output_pair_file_path, 'wb') as fw:
        pickle.dump(new_pair_list, fw)
        
    print(f"[FINISH] Integrated history saved to: {output_hist_file_path}")
    print(f"[FINISH] Integrated pairs saved to: {output_pair_file_path}")
    
    return

In [ ]:
import os
import pickle

def check_editors_edits_pages_hyperlinks(BASE_DIR, threshold):
    """
    Displays summary statistics for the filtered dataset based on the given threshold.

    Args:
        BASE_DIR (str): The root directory of the project.
        threshold (int): The edit threshold used for filtering.

    Returns:
        None: Prints the count of editors, total edits, unique pages, and hyperlinks.
    """
    # Define standardized directory paths for data access
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    TARGET_DIR = os.path.join(DATA_DIR, 'conditioned_history')
    
    # 1. Load the conditioned history and pair list using standardized path naming
    hist_file_name = f'condi_pages_{threshold}_editors_{threshold}.pickle'
    hist_file_path = os.path.join(TARGET_DIR, hist_file_name)
    with open(hist_file_path, 'rb') as fr:
        conditioned_history = pickle.load(fr)
        
    pair_file_name = f'condi_pairs_{threshold}_{threshold}.pickle'
    pair_file_path = os.path.join(TARGET_DIR, pair_file_name)
    with open(pair_file_path, 'rb') as fr:
        pair_list = pickle.load(fr)
        
    # 2. Calculate statistics
    editors = list(conditioned_history.keys())
    total_edits = 0
    unique_pages = set()

    for editor in editors:
        events = conditioned_history[editor]
        total_edits += len(events)
        for event in events:
            # event[1] is the page_id
            unique_pages.add(event[1])

    # 3. Print the summary results
    print(f"Summary Statistics for Threshold: {threshold}")
    print(f"----------------------------------------")
    print(f"Total Editors: {len(editors):,}")
    print(f"Total Edits:   {total_edits:,}")
    print(f"Unique Pages:  {len(unique_pages):,}")
    print(f"Hyperlinks:    {len(pair_list):,}")
    print(f"----------------------------------------")

    return